# Predicting Enzyme Functions from Protein Sequences

Frozen ESM-2 features, simple prediction heads, and an audit of the evaluation data.

Author: Younis Anis, AI Engineer


## Philosophy of this work

> *Frozen protein-language-model embeddings, computed once and cached to persistent storage;
> evaluated only on the official identity-controlled held-out splits, whose leakage we measure
> rather than assume; with the task modelled as it actually is — hierarchical and multi-label —
> and reported per EC level and per class-frequency bin rather than as a single aggregate;
> under a strict free-tier compute budget with mandatory checkpoint/resume; reproducing the
> project's own prior results before replacing them, and stating plainly what each number can
> and cannot support.*

## How to run

Install `requirements.txt` in your Python environment and open this notebook from the cloned project or its `notebooks/` directory. In Colab, place the project under the project root or set `ECPRED_ROOT` to its mounted location before Stage 1.

Leave `SMOKE_TEST = True` for a first pass on small subsets. Smoke outputs go under `outputs/smoke/`; set it to `False` for the main run. Encoding still requires model weights and is much slower without a GPU.


## Known data quirks

The supplied files contain schema differences that the reader handles in memory. The original data bytes are preserved.

| # | Finding |
|---|---|
| N1 | Train and test headers use different names. The split50 train header also splits `EC number` into two fields above three-field data rows. The reader repairs that known header, then the notebook normalizes columns by position. |
| N2 | Split CSVs use CRLF line endings. These are handled by the reader and checked after loading. |
| N3 | `split30_test_split_0_curate.csv` wraps each entire TSV line in double quotes. The tested reader handles this format. |
| N4 | Fold-0 exact test-in-train sequence overlap is zero for splits 10/30/50/70 and 9,611 rows (21.29%) for split100. Against the historical CD-HIT-90 training FASTA it is 4,417 (9.78%). |
| N5 | All test EC labels appear in the full supplied training label space. This does not test unseen-EC generalization. |
| N6 | No supplied sequence exceeds 1,022 residues, matching the configured encoding limit. |
| N7 | Semicolon-separated labels are parsed as sets of individual EC activities. |
| N8 | The CD-HIT FASTA header format is `>{Entry}|{EC label string}`; fold 0 has 108,279 records. This historical branch is separate from the main split10–70 experiments. |
| N9 | The supplied benchmark includes 50 split CSVs, five CD-HIT FASTAs and five pretrained CLEAN artifacts. Restore the required inputs before running the notebook. |


## Stage 0 — Configuration

Every switch in the notebook lives in the next cell, so a run is changed by editing one place only.

In [ ]:
# =====================================================================
#  STAGE 0 - CONFIGURATION.  This is the only cell you normally edit.
# =====================================================================

# ---------------------------------------------------------------------
#  SMOKE_TEST
#  ----------
#  True  -> the WHOLE notebook runs end to end on a tiny subset
#           (a few hundred sequences per split, a few epochs, ~5 minutes)
#           and every artefact is written under  outputs/smoke/  so a
#           smoke run can NEVER overwrite or contaminate a real result.
#  False -> the real run.
#  Run it once with True, confirm every stage prints OK, then set False.
# ---------------------------------------------------------------------
SMOKE_TEST            = True

# --- scope -----------------------------------------------------------
SPLITS_TO_RUN         = (10, 30, 50, 70)   # identity thresholds to embed and model
FOLD                  = 0                  # fold 0 only for this run
RUN_SPLIT100          = False              # gated: the expensive fifth curve point.
                                           # The split100 CPU-only work (leakage audit +
                                           # legacy forensic reproduction) runs regardless.

# --- encoder ---------------------------------------------------------
ESM_MODEL_ID          = "facebook/esm2_t33_650M_UR50D"   # 33 layers, 1280-d
EMB_TAG               = "esm2_650M"
SECOND_PASS_150M      = False                            # wired, off by default
ESM_MODEL_ID_150M     = "facebook/esm2_t30_150M_UR50D"   # 640-d, CPU-friendly
EMB_TAG_150M          = "esm2_150M"

MAX_LEN               = 1022    # residues. Lossless on this data (N6). Never 512.
MAX_TOKENS_BATCH      = 16384   # token budget per batch; halved automatically on CUDA OOM
SHARD_SIZE            = 2000    # sequences per checkpoint shard written to Drive
KEEP_SHARDS           = False   # keep shard files after a validated merge?

# --- stages ----------------------------------------------------------
RUN_LEGACY_FORENSIC   = True    # Stage 4: reproduce the 54.5%-leaked legacy split
RUN_AAC_BASELINE      = True    # Stage 5: honest AAC + LinearSVC on the official test sets
RUN_DIAMOND           = True    # Stage 11: DIAMOND homology-transfer baseline
FORCE_RECOMPUTE       = False   # ignore every cached result and redo the work

# --- prediction heads ------------------------------------------------
HEAD_EPOCHS           = 60
HEAD_BATCH            = 512
HEAD_LR               = 1e-3
HEAD_WEIGHT_DECAY     = 1e-4
HEAD_HIDDEN           = 1024
HEAD_DROPOUT          = 0.3
EARLY_STOP_PATIENCE   = 5
VAL_FRACTION          = 0.10    # validation slice carved out of TRAIN, never out of test
EVAL_CHUNK            = 4096    # rows scored at a time; keeps the score matrix small

SEED                  = 42

# --- smoke-test sizes (ignored when SMOKE_TEST is False) --------------
SMOKE_N_TRAIN         = 300
SMOKE_N_TEST          = 100
SMOKE_EPOCHS          = 3
SMOKE_FASTA_N         = 3000

In [ ]:
# Derived configuration and the smoke-test banner.
AUDIT_SPLITS = tuple(sorted(set(SPLITS_TO_RUN) | {100}))          # split100 audit is CPU-only
EMBED_SPLITS = tuple(SPLITS_TO_RUN) + ((100,) if RUN_SPLIT100 else ())
EFFECTIVE_EPOCHS = SMOKE_EPOCHS if SMOKE_TEST else HEAD_EPOCHS

MODEL_PASSES = [(ESM_MODEL_ID, EMB_TAG)]
if SECOND_PASS_150M:
    MODEL_PASSES.append((ESM_MODEL_ID_150M, EMB_TAG_150M))

_bar = "#" * 78
if SMOKE_TEST:
    print(_bar)
    print("#   SMOKE TEST MODE IS ON".ljust(77) + "#")
    print(("#   tiny subsets ({} train / {} test per split), {} epochs"
           .format(SMOKE_N_TRAIN, SMOKE_N_TEST, SMOKE_EPOCHS)).ljust(77) + "#")
    print("#   everything is written under  outputs/smoke/  and NOTHING real is touched".ljust(77) + "#")
    print("#   set SMOKE_TEST = False in Stage 0 for the real run".ljust(77) + "#")
    print(_bar)
else:
    print(_bar)
    print("#   FULL RUN - results are written to outputs/ on Drive".ljust(77) + "#")
    print(_bar)

print("splits to embed :", EMBED_SPLITS)
print("splits to audit :", AUDIT_SPLITS)
print("encoder passes  :", [t for _, t in MODEL_PASSES])
print("epochs          :", EFFECTIVE_EPOCHS)

## Stage 1 — Environment

Find the project using its `.ecpred_root` marker and load `src/ecpaths.py`. Local notebooks search the current directory and its parents. `ECPRED_ROOT` can select another project location. In Colab, the bootstrap can mount Drive and find a marked project directly under MyDrive.


In [ ]:
import os
import sys
from pathlib import Path

def _find_notebook_root():
    hint = os.environ.get("ECPRED_ROOT")
    if hint:
        root = Path(hint).expanduser().resolve()
        if not (root / ".ecpred_root").is_file():
            raise FileNotFoundError("ECPRED_ROOT must point to a directory containing .ecpred_root")
        return root
    here = Path.cwd().resolve()
    for root in (here, *here.parents):
        if (root / ".ecpred_root").is_file():
            return root
    try:
        from google.colab import drive
    except ImportError:
        raise FileNotFoundError(
            "Open the notebook from the project directory, or set ECPRED_ROOT to it."
        ) from None
    drive.mount("/content/drive")
    drive_root = Path("/content/drive/MyDrive")
    candidates = sorted({marker.parent for marker in drive_root.glob("*/.ecpred_root")})
    if (drive_root / ".ecpred_root").is_file():
        candidates.append(drive_root)
    if len(candidates) != 1:
        raise FileNotFoundError(
            "Expected one marked project under MyDrive. Set ECPRED_ROOT to the desired project "
            "after mounting Drive. Found: {}".format([str(p) for p in candidates])
        )
    return candidates[0]

PROJECT_ROOT = _find_notebook_root()
if not (PROJECT_ROOT / "src" / "ecpaths.py").is_file():
    raise FileNotFoundError("Restore src/ecpaths.py in the selected project directory.")
os.environ["ECPRED_ROOT"] = str(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import ecpaths as P
P.verify(strict=True)


Next we import the scientific stack and use the repository's read-only CSV reader. It repairs the split50 header in memory and handles the quoted split30 file without changing the raw datasets. Install missing Python dependencies from `requirements.txt` before running; this notebook does not install or downgrade packages automatically.


In [ ]:
import os, sys, gc, csv, json, math, time, random, hashlib, shutil, subprocess, tempfile
from pathlib import Path

try:
    import numpy as np
    import pandas as pd
    import scipy.sparse as sp
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Install the project requirements.txt in this notebook kernel first.") from exc

P.ensure_output_dirs()

# ---------------------------------------------------------------------
# 1a. Keep the parser in one tested source module.
# ---------------------------------------------------------------------
read_split_csv = P.read_split_csv
print("safe CSV reader : ecpaths.read_split_csv")

# ---------------------------------------------------------------------
# 1b. Smoke-test output redirection.
#     Every artefact path is built from ecpaths and then, in smoke mode,
#     rewritten under  <OUTPUTS>/smoke/  keeping the same sub-structure.
# ---------------------------------------------------------------------
SMOKE_ROOT = P.OUTPUTS / "smoke"

def _redirect(p: Path) -> Path:
    p = Path(p)
    if not SMOKE_TEST:
        p.parent.mkdir(parents=True, exist_ok=True)
        return p
    q = SMOKE_ROOT / p.relative_to(P.OUTPUTS)
    q.parent.mkdir(parents=True, exist_ok=True)
    return q

def res_path(name: str) -> Path:
    return _redirect(P.result_file(name))

def emb_path(split: int, fold: int, part: str, tag: str) -> Path:
    return _redirect(P.emb_file(split, fold, part, model=tag))

def ids_path(split: int, fold: int, part: str, tag: str) -> Path:
    return _redirect(P.ids_file(split, fold, part, model=tag))

def fig_path(name: str) -> Path:
    return _redirect(P.FIGURES / name)

RESULTS_DIR = res_path("_").parent
EMB_DIR = emb_path(10, 0, "train", EMB_TAG).parent
FIGURES_DIR = fig_path("_").parent
print("results dir     :", RESULTS_DIR)
print("embeddings dir  :", EMB_DIR)
print("figures dir     :", FIGURES_DIR)

# ---------------------------------------------------------------------
# 1c. Reproducibility and bookkeeping.
# ---------------------------------------------------------------------
def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)

set_seed(SEED)

INPUT_FILES = set()        # every input file actually read -> hashed into the manifest
STAGE_TIMES = {}           # stage name -> wall-clock seconds

class stage_timer:
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        self.t0 = time.time(); print("\n>>> {} ...".format(self.name)); return self
    def __exit__(self, *exc):
        dt = time.time() - self.t0
        STAGE_TIMES[self.name] = round(dt, 2)
        print("<<< {} finished in {:.1f}s".format(self.name, dt))
        return False

def cached_json(name, builder, force=None):
    """Run builder() once, cache the result as JSON, reuse it on any later run."""
    force = FORCE_RECOMPUTE if force is None else force
    path = res_path(name)
    if path.exists() and not force:
        print("cached  ->", path.name)
        return json.loads(path.read_text())
    obj = builder()
    path.write_text(json.dumps(obj, indent=2, default=str))
    print("written ->", path.name)
    return obj

In [ ]:
# 1d. Check dependencies without downloading packages or importing torch early.
import importlib.util

_needed = {"torch": "torch", "transformers": "transformers",
           "sklearn": "scikit-learn", "scipy": "scipy", "matplotlib": "matplotlib"}
_missing = [pipname for mod, pipname in _needed.items()
            if importlib.util.find_spec(mod) is None]
if _missing:
    raise ModuleNotFoundError(
        "Missing packages: {}. Install requirements.txt in this notebook kernel first."
        .format(", ".join(_missing))
    )
print("all required packages are available")


In [ ]:
# 1e. Use persistent project-local caches before importing torch / transformers.
MODEL_CACHE = P.OUTPUTS / "model_cache"          # never redirected: shared by smoke and real runs
(MODEL_CACHE / "hf").mkdir(parents=True, exist_ok=True)
os.environ["TORCH_HOME"] = str(MODEL_CACHE)
os.environ["HF_HOME"] = str(MODEL_CACHE / "hf")
os.environ["TRANSFORMERS_CACHE"] = str(MODEL_CACHE / "hf")

import torch
import sklearn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)

def _ram_gb():
    try:
        return os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
    except Exception:
        return float("nan")

ENV_REPORT = {
    "python": sys.version.split()[0],
    "numpy": np.__version__, "pandas": pd.__version__, "scipy": sp.__version__ if hasattr(sp, "__version__") else "",
    "scikit_learn": sklearn.__version__, "torch": torch.__version__,
    "matplotlib": matplotlib.__version__,
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "none",
    "gpu_total_mem_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if DEVICE == "cuda" else 0.0,
    "system_ram_gb": round(_ram_gb(), 2),
}
try:
    import transformers
    ENV_REPORT["transformers"] = transformers.__version__
except Exception as _e:
    ENV_REPORT["transformers"] = "IMPORT FAILED: {}".format(_e)

_usage = shutil.disk_usage(P.ROOT)
ENV_REPORT["drive_free_gb"] = round(_usage.free / 1e9, 2)

for k, v in ENV_REPORT.items():
    print("{:<18}: {}".format(k, v))

if DEVICE != "cuda":
    print("\nWARNING: no GPU is attached. Stage 6 will be extremely slow.")
    print("         Use a GPU-enabled Python/Colab runtime before a full run.")
if ENV_REPORT["drive_free_gb"] < 2.0:
    print("\nWARNING: less than 2 GB free at the project root. Plan space for embeddings and model weights.")

## Stage 2 — Loading and schema normalisation

Train and test CSVs disagree on column names (N1), so columns are renamed **by position** into `id`, `ec`, `seq`.
Peak RAM ≈ 1.5 GB; nothing here is cached, because reading is cheap compared with the checks it enables.

In [ ]:
import re
_AA_RE = re.compile(r"^[A-Z]+$")
STANDARD_AA = "ACDEFGHIKLMNPQRSTVWY"

_SPLIT_CACHE = {}

def _positional_normalise(df, source):
    """N1: rename by POSITION, never by name. col0 -> id, col1 -> ec, col2 -> seq."""
    assert df.shape[1] == 3, "{}: expected 3 columns, got {} ({})".format(source, df.shape[1], list(df.columns))
    df = df.copy()
    df.columns = ["id", "ec", "seq"]
    for c in ("id", "ec", "seq"):
        df[c] = df[c].astype(str).str.strip()
    return df

def validate_split_frame(df, source, split, fold, part, expect_rows=None):
    """Assertions that must fail loudly - silent schema failures are limitation L11."""
    n = len(df)
    if expect_rows is not None:
        assert n == expect_rows, "{}: expected {} rows, got {}".format(source, expect_rows, n)
    assert df["ec"].notna().all() and (df["ec"] != "").all(), "{}: empty EC label found".format(source)
    assert df["seq"].notna().all() and (df["seq"] != "").all(), "{}: empty sequence found".format(source)
    bad_quote = df.apply(lambda col: col.str.contains('"', regex=False).sum()).sum()
    bad_cr = df.apply(lambda col: col.str.contains("\r", regex=False).sum()).sum()
    assert bad_quote == 0, '{}: {} cells still contain a double quote'.format(source, bad_quote)
    assert bad_cr == 0, "{}: {} cells still contain a carriage return".format(source, bad_cr)
    bad_seq = (~df["seq"].str.match(_AA_RE)).sum()
    assert bad_seq == 0, "{}: {} sequences contain non A-Z characters".format(source, bad_seq)
    maxlen = int(df["seq"].str.len().max())
    assert maxlen <= MAX_LEN, "{}: max sequence length {} exceeds MAX_LEN {}".format(source, maxlen, MAX_LEN)
    return {"rows": n, "max_len": maxlen}

def _smoke_subset(tr, te):
    """Keep the closed-set property (N5) inside the smoke subset."""
    rng = np.random.RandomState(SEED)
    tr_idx = rng.choice(len(tr), size=min(SMOKE_N_TRAIN, len(tr)), replace=False)
    tr_s = tr.iloc[np.sort(tr_idx)].reset_index(drop=True)
    train_ecs = set()
    for s in tr_s["ec"]:
        train_ecs.update(parse_ec_set(s))
    keep = te["ec"].map(lambda s: parse_ec_set(s) <= train_ecs and len(parse_ec_set(s)) > 0)
    te_ok = te[keep]
    if len(te_ok) == 0:
        te_ok = te.head(SMOKE_N_TEST)
    te_s = te_ok.head(SMOKE_N_TEST).reset_index(drop=True)
    return tr_s, te_s

def parse_ec_set(label_string):
    """Split a composite EC label on ';' into the set of true EC numbers (N7)."""
    return {e.strip() for e in str(label_string).split(";") if e.strip()}

def load_split(split, fold=FOLD, part="train", raw=False):
    """Read one official split CSV through the safe reader and normalise it positionally.

    raw=True returns the FULL frame even in smoke mode (needed by the audit stage).
    """
    key = (split, fold, part, raw)
    if key in _SPLIT_CACHE:
        return _SPLIT_CACHE[key]
    path = P.train_csv(split, fold) if part == "train" else P.test_csv(split, fold)
    INPUT_FILES.add(str(path))
    df = _positional_normalise(read_split_csv(path), path.name)
    exp = None
    if (split, fold) in P.EXPECTED_ROWS:
        exp = P.EXPECTED_ROWS[(split, fold)][0 if part == "train" else 1]
    validate_split_frame(df, path.name, split, fold, part, expect_rows=exp)
    if raw or not SMOKE_TEST:
        _SPLIT_CACHE[key] = df
        return df
    # smoke mode: subset train and test together so the label space stays closed
    tr = _positional_normalise(read_split_csv(P.train_csv(split, fold)), "train")
    te = _positional_normalise(read_split_csv(P.test_csv(split, fold)), "test")
    tr_s, te_s = _smoke_subset(tr, te)
    _SPLIT_CACHE[(split, fold, "train", False)] = tr_s
    _SPLIT_CACHE[(split, fold, "test", False)] = te_s
    del tr, te; gc.collect()
    return _SPLIT_CACHE[key]

def cache_clear(keep=()):
    keep = set(keep)
    for k in list(_SPLIT_CACHE):
        if k[0] not in keep:
            del _SPLIT_CACHE[k]
    gc.collect()

In [ ]:
with stage_timer("Stage 2 - load and validate every split"):
    rows = []
    for s in AUDIT_SPLITS:
        for part in ("train", "test"):
            df = load_split(s, FOLD, part, raw=True)
            nonstd = int(df["seq"].map(lambda q: sum(ch not in STANDARD_AA for ch in q)).sum())
            rows.append({"split": s, "part": part, "rows": len(df),
                         "residues": int(df["seq"].str.len().sum()),
                         "mean_len": round(float(df["seq"].str.len().mean()), 1),
                         "max_len": int(df["seq"].str.len().max()),
                         "non_standard_aa": nonstd})
    schema_table = pd.DataFrame(rows)
    print(schema_table.to_string(index=False))

    # N3 - the one whole-line-quoted file must parse to (1648, 3) with clean cells.
    if 30 in AUDIT_SPLITS:
        q = load_split(30, 0, "test", raw=True)
        assert q.shape == (1648, 3), "split30 fold-0 test parsed to {} - expected (1648, 3)".format(q.shape)
        print("\nN3 OK : split30_test_split_0_curate.csv -> (1648, 3), zero stray quote characters")
    print("Stage 2 OK : positional normalisation and every integrity assertion passed")

## Stage 3 — Dataset characterisation and leakage audit

This stage measures, rather than assumes, how much of each official test set is already present in its own training set.
It is CPU-only and runs for split100 even when `RUN_SPLIT100` is off, because that is where the headline finding lives. Peak RAM ≈ 2.5 GB.

In [ ]:
def read_cdhit_fasta(fold=0, limit=None):
    """Minimal FASTA reader for the CD-HIT output. Header format is >{Entry}|{EC} (N8)."""
    path = P.cdhit_fasta(fold)
    INPUT_FILES.add(str(path))
    ids, ecs, seqs = [], [], []
    cur_id, cur_ec, buf = None, None, []
    with open(path, "r") as f:
        for line in f:
            line = line.rstrip("\n").rstrip("\r")
            if line.startswith(">"):
                if cur_id is not None:
                    ids.append(cur_id); ecs.append(cur_ec); seqs.append("".join(buf))
                    if limit and len(ids) >= limit:
                        return pd.DataFrame({"id": ids, "ec": ecs, "seq": seqs})
                head = line[1:]
                parts = head.split("|")
                cur_id, cur_ec, buf = parts[0], (parts[1] if len(parts) > 1 else ""), []
            else:
                buf.append(line)
    if cur_id is not None:
        ids.append(cur_id); ecs.append(cur_ec); seqs.append("".join(buf))
    return pd.DataFrame({"id": ids, "ec": ecs, "seq": seqs})


def audit_one_split(split):
    tr = load_split(split, FOLD, "train", raw=True)
    te = load_split(split, FOLD, "test", raw=True)

    tr_ec, te_ec = set(), set()
    tr_multi = te_multi = 0
    for s in tr["ec"]:
        e = parse_ec_set(s); tr_ec |= e; tr_multi += (len(e) > 1)
    for s in te["ec"]:
        e = parse_ec_set(s); te_ec |= e; te_multi += (len(e) > 1)

    train_seqs = set(tr["seq"])
    leaked_mask = te["seq"].isin(train_seqs).to_numpy()
    n_leak = int(leaked_mask.sum())

    # EC agreement on the leaked rows, counted BOTH ways (see plan N4).
    agree_raw = agree_set = 0
    if n_leak:
        lut = {}
        for s, e in zip(tr["seq"], tr["ec"]):
            lut.setdefault(s, e)
        for s, e in zip(te["seq"][leaked_mask], te["ec"][leaked_mask]):
            t = lut[s]
            agree_raw += (t == e)
            agree_set += (parse_ec_set(t) == parse_ec_set(e))

    lvl1 = pd.Series([e.split(".")[0] for s in tr["ec"] for e in parse_ec_set(s)]).value_counts().sort_index()

    out = {
        "split": split,
        "train_rows": len(tr), "test_rows": len(te),
        "train_residues": int(tr["seq"].str.len().sum()),
        "test_residues": int(te["seq"].str.len().sum()),
        "train_label_strings": int(tr["ec"].nunique()), "test_label_strings": int(te["ec"].nunique()),
        "train_unique_ec": len(tr_ec), "test_unique_ec": len(te_ec),
        "train_multi_label_rows": int(tr_multi), "test_multi_label_rows": int(te_multi),
        "unseen_test_ec": len(te_ec - tr_ec),
        "test_rows_with_unseen_ec": int(te["ec"].map(lambda s: bool(parse_ec_set(s) - tr_ec)).sum()),
        "leaked_rows": n_leak,
        "leaked_pct": round(100.0 * n_leak / max(len(te), 1), 2),
        "leaked_ec_agree_raw_string": agree_raw,
        "leaked_ec_agree_set_equality": agree_set,
        "dedup_rows": int(len(te) - n_leak),
        "train_level1_distribution": {k: int(v) for k, v in lvl1.items()},
    }
    np.save(res_path("dedup_mask_split{}_fold{}.npy".format(split, FOLD)), ~leaked_mask)
    del tr, te, train_seqs; gc.collect()
    return out


def build_stage3():
    audit = {"per_split": [audit_one_split(s) for s in AUDIT_SPLITS]}
    if 100 in AUDIT_SPLITS:
        cd = read_cdhit_fasta(FOLD, limit=SMOKE_FASTA_N if SMOKE_TEST else None)
        te = load_split(100, FOLD, "test", raw=True)
        cd_seqs = set(cd["seq"])
        n = int(te["seq"].isin(cd_seqs).sum())
        audit["cdhit"] = {"records": len(cd),
                          "test_rows_in_cdhit_train": n,
                          "pct": round(100.0 * n / len(te), 2)}
        del cd, cd_seqs, te; gc.collect()
    return audit

with stage_timer("Stage 3 - characterisation and leakage audit"):
    STAGE3 = cached_json("stage3_data_audit.json", build_stage3)
    a = pd.DataFrame(STAGE3["per_split"]).drop(columns=["train_level1_distribution"])
    print(a.to_string(index=False))
    if "cdhit" in STAGE3:
        print("\nCD-HIT'd split100 train :", STAGE3["cdhit"])

In [ ]:
# Stage 3 assertions. In smoke mode the numbers are subset-dependent, so they are reported only.
with stage_timer("Stage 3 - assertions"):
    by = {d["split"]: d for d in STAGE3["per_split"]}
    if SMOKE_TEST:
        print("SMOKE MODE: exact-count assertions skipped (subsets change every count).")
        for s, d in by.items():
            print("  split{:<4} leaked={:<6} unseen_ec={}".format(s, d["leaked_rows"], d["unseen_test_ec"]))
    else:
        for s, d in by.items():
            assert d["unseen_test_ec"] == 0, \
                "split{}: {} test EC numbers are unseen in train - N5 violated".format(s, d["unseen_test_ec"])
        for s in (10, 30, 50, 70):
            if s in by:
                assert by[s]["leaked_rows"] == 0, \
                    "split{}: expected 0 verbatim leaked rows, measured {}".format(s, by[s]["leaked_rows"])
        if 100 in by:
            assert by[100]["leaked_rows"] == 9611, \
                "split100: expected 9,611 leaked rows, measured {}".format(by[100]["leaked_rows"])
            assert abs(by[100]["leaked_pct"] - 21.29) < 0.02
        if "cdhit" in STAGE3:
            assert STAGE3["cdhit"]["records"] == 108279, \
                "CD-HIT fold-0 FASTA: expected 108,279 records, got {}".format(STAGE3["cdhit"]["records"])
            assert STAGE3["cdhit"]["test_rows_in_cdhit_train"] == 4417, \
                "expected 4,417 test rows inside the CD-HIT train set, got {}".format(
                    STAGE3["cdhit"]["test_rows_in_cdhit_train"])
        print("Stage 3 OK : leakage profile, closed-set property and CD-HIT counts all as expected")

**What this table supports and what it does not.** It supports the statement that CLEAN's identity-clustered splits 10–70 are free of verbatim sequence leakage while the unclustered split100 is not, and that CD-HIT-90 clustering halves that leakage.
It does not measure near-duplicate leakage below 100% identity, so it is a lower bound on contamination rather than a complete audit.

## Stage 4 — Forensic reproduction of the legacy evaluation bug

Run A reproduces the original leaked split exactly, and Run B applies the corrected per-group split that was written but never executed.
Reproducing the earlier number before replacing it is what makes the later numbers trustworthy. Peak RAM ≈ 2 GB.

In [ ]:
def get_aac(seq):
    """Amino-acid composition, 20 features. Reused unchanged from Clean_SVM.ipynb cell 0."""
    seq = str(seq).upper()
    n = len(seq)
    if n == 0:
        return np.zeros(20, dtype=np.float32)
    return np.array([seq.count(a) / n for a in STANDARD_AA], dtype=np.float32)

def aac_matrix(seqs):
    X = np.empty((len(seqs), 20), dtype=np.float32)
    for i, s in enumerate(seqs):
        X[i] = get_aac(s)
    return X

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

def build_stage4():
    cd = read_cdhit_fasta(FOLD, limit=SMOKE_FASTA_N if SMOKE_TEST else None)
    if not SMOKE_TEST:
        assert len(cd) == 108279, "CD-HIT fold-0 FASTA: expected 108,279 records, got {}".format(len(cd))

    # The legacy pipeline treated the composite label string as one atomic class.
    counts = cd["ec"].value_counts()
    min_n = 5 if SMOKE_TEST else 20
    keep = counts[counts >= min_n].index
    df_filtered = cd[cd["ec"].isin(keep)]          # index deliberately NOT reset (this is the bug)
    n_classes = int(df_filtered["ec"].nunique())
    if not SMOKE_TEST:
        assert n_classes == 791, "expected 791 classes with >= 20 sequences, got {}".format(n_classes)

    le = LabelEncoder().fit(df_filtered["ec"])
    X_all = aac_matrix(df_filtered["seq"].tolist())
    X_all = pd.DataFrame(X_all, index=df_filtered.index)

    # ---------------- Run A : the original bug, reproduced literally ----------------
    # Equivalent to the legacy  groupby('EC').apply(sample).reset_index(drop=True),
    # written explicitly so the row selection does not depend on the pandas version.
    _parts = [gdf.sample(min(len(gdf), 100), random_state=SEED)
              for _, gdf in df_filtered.groupby("ec", sort=True)]
    df_sampled = pd.concat(_parts).reset_index(drop=True)
    df_remaining = df_filtered[~df_filtered.index.isin(df_sampled.index)]

    def _fit_eval(Xtr, ytr, Xte, yte):
        if len(Xtr) == 0 or len(Xte) == 0:
            print("  WARNING: an empty train or test partition - metrics not computable here.")
            return None, None
        clf = LinearSVC(random_state=SEED)
        clf.fit(Xtr, ytr)
        pred = clf.predict(Xte)
        return (float(accuracy_score(yte, pred)),
                float(f1_score(yte, pred, average="macro", zero_division=0)))

    accA, f1A = _fit_eval(aac_matrix(df_sampled["seq"].tolist()),
                          le.transform(df_sampled["ec"]),
                          X_all.loc[df_remaining.index].to_numpy(),
                          le.transform(df_remaining["ec"]))

    # How much of that "independent test set" is literally training data?
    sampled_keys = set(zip(df_sampled["id"], df_sampled["ec"]))
    leaked = int(sum((i, e) in sampled_keys for i, e in zip(df_remaining["id"], df_remaining["ec"])))

    # ---------------- Run B : the corrected per-group split ----------------
    tr_idx, te_idx = [], []
    for _, g in df_filtered.groupby("ec"):
        take = g.sample(min(len(g), 100), random_state=SEED)
        tr_idx.extend(take.index.tolist())
        te_idx.extend(g.index.difference(take.index).tolist())
    dfB_tr = df_filtered.loc[tr_idx]
    dfB_te = df_filtered.loc[te_idx]

    accB, f1B = _fit_eval(X_all.loc[dfB_tr.index].to_numpy(), le.transform(dfB_tr["ec"]),
                          X_all.loc[dfB_te.index].to_numpy(), le.transform(dfB_te["ec"]))

    out = {
        "fasta_records": int(len(cd)), "classes_kept": n_classes,
        "run_a": {"train_rows": int(len(df_sampled)), "test_rows": int(len(df_remaining)),
                  "test_classes": int(df_remaining["ec"].nunique()),
                  "accuracy": accA, "macro_f1": f1A,
                  "test_rows_that_are_training_rows": leaked,
                  "leak_pct": round(100.0 * leaked / max(len(df_remaining), 1), 2)},
        "run_b": {"train_rows": int(len(dfB_tr)), "test_rows": int(len(dfB_te)),
                  "test_classes": int(dfB_te["ec"].nunique()),
                  "accuracy": accB, "macro_f1": f1B},
    }
    del cd, df_filtered, X_all; gc.collect()
    return out

if RUN_LEGACY_FORENSIC:
    with stage_timer("Stage 4 - legacy-bug forensic reproduction"):
        STAGE4 = cached_json("stage4_legacy_forensic.json", build_stage4)
        print(json.dumps(STAGE4, indent=2))
        if SMOKE_TEST:
            print("\nSMOKE MODE: the 0.27778 fidelity assertion is skipped on a subset.")
        else:
            got = STAGE4["run_a"]["accuracy"]
            assert got is not None, "FORENSIC REPRODUCTION FAILED: Run A produced no test partition."
            assert abs(got - 0.27778) <= 1e-3, (
                "FORENSIC REPRODUCTION FAILED: Run A accuracy is {:.5f}, expected 0.27778 +/- 0.001. "
                "The pipeline does not faithfully reproduce the legacy result, so no later number "
                "from it should be trusted until this is explained.".format(got))
            print("\nRun A reproduces the legacy accuracy 0.27778 to within 1e-3 - reproduction is faithful.")
            print("Run A 'independent' test set is {}% training data."
                  .format(STAGE4["run_a"]["leak_pct"]))
else:
    STAGE4 = None
    print("Stage 4 skipped (RUN_LEGACY_FORENSIC = False)")

**What this supports.** The saved forensic results record 0.27778 accuracy for the leaked split and about 0.29932 for the corrected internal split. These are different test populations: the corrected split leaves 40,459 rows and 297 test classes, compared with 47,979 rows and 775 classes in Run A. The change is not a fair model improvement. The main comparison instead uses the supplied held-out test files and treats composite annotations as label sets.


## Stage 8 (defined here) — Label space, EC-level maps and evaluation machinery

These helpers are defined before Stage 5 because the composition baseline also uses them. Targets are sparse multi-label matrices. Each broader EC-level score is the maximum over its descendant full-EC scores. Thresholds are selected independently per level, so final binary calls are not guaranteed to be jointly nested across levels.


In [ ]:
# ---------------------------------------------------------------------
# Label space, hierarchy roll-up maps and sparse multi-hot targets.
# ---------------------------------------------------------------------
def _prefix(ec, level):
    parts = str(ec).split(".")
    while len(parts) < 4:
        parts.append("-")
    return ".".join(parts[:level])

def build_level_maps(labels):
    """For each level, the grouping of level-4 labels needed for a max roll-up."""
    maps = {}
    L = len(labels)
    for lv in (1, 2, 3, 4):
        keys = [_prefix(x, lv) for x in labels]
        names = sorted(set(keys))
        kidx = {k: i for i, k in enumerate(names)}
        idx = np.array([kidx[k] for k in keys], dtype=np.int64)
        perm = np.argsort(idx, kind="stable")
        starts = np.searchsorted(idx[perm], np.arange(len(names)), side="left")
        M = sp.csr_matrix((np.ones(L, dtype=np.uint8), (np.arange(L), idx)), shape=(L, len(names)))
        maps[lv] = {"names": names, "idx": idx, "perm": perm, "starts": starts, "M": M}
    return maps

def apply_level(matrix, m):
    """Roll a (rows, |L|) matrix up to (rows, n_level_labels) by taking the max over children."""
    return np.maximum.reduceat(matrix[:, m["perm"]], m["starts"], axis=1)

def build_targets(ec_strings, label_index):
    rows, cols = [], []
    for i, s in enumerate(ec_strings):
        for e in parse_ec_set(s):
            j = label_index.get(e)
            if j is not None:
                rows.append(i); cols.append(j)
    data = np.ones(len(rows), dtype=np.uint8)
    return sp.csr_matrix((data, (rows, cols)), shape=(len(ec_strings), len(label_index)))

BIN_EDGES = [("N>=100", 100, 10**12), ("20-99", 20, 99), ("5-19", 5, 19), ("1-4", 1, 4)]

def frequency_bins(freq):
    out = np.full(len(freq), -1, dtype=np.int64)
    for b, (_, lo, hi) in enumerate(BIN_EDGES):
        out[(freq >= lo) & (freq <= hi)] = b
    return out

_LABEL_SPACE = {}

def get_label_space(split):
    """Closed label space taken from TRAIN only (N5 guarantees full test coverage)."""
    if split in _LABEL_SPACE:
        return _LABEL_SPACE[split]
    tr = load_split(split, FOLD, "train")
    te = load_split(split, FOLD, "test")
    labels = sorted({e for s in tr["ec"] for e in parse_ec_set(s)})
    index = {e: i for i, e in enumerate(labels)}
    Ytr = build_targets(tr["ec"].tolist(), index)
    Yte = build_targets(te["ec"].tolist(), index)
    maps = build_level_maps(labels)
    freq = {lv: np.asarray(((Ytr @ maps[lv]["M"]) > 0).sum(0)).ravel().astype(np.int64) for lv in (1, 2, 3, 4)}
    bins = {lv: frequency_bins(freq[lv]) for lv in (1, 2, 3, 4)}
    is_multi = np.asarray(Ytr.sum(1)).ravel() > 1
    space = {"split": split, "labels": labels, "index": index, "maps": maps,
             "Ytr": Ytr, "Yte": Yte, "freq": freq, "bins": bins, "is_multi": is_multi,
             "n_train": Ytr.shape[0], "n_test": Yte.shape[0], "n_labels": len(labels)}
    res_path("label_space_split{}_fold{}.json".format(split, FOLD)).write_text(
        json.dumps({"n_labels": len(labels), "n_train": Ytr.shape[0], "n_test": Yte.shape[0],
                    "labels": labels}, indent=1))
    _LABEL_SPACE[split] = space
    return space

def train_val_indices(space, seed=SEED):
    """10% validation slice carved out of TRAIN, stratified by multi-label status."""
    rng = np.random.RandomState(seed)
    n = space["n_train"]
    tr_idx, va_idx = [], []
    for flag in (False, True):
        pool = np.where(space["is_multi"] == flag)[0]
        rng.shuffle(pool)
        k = max(1, int(round(VAL_FRACTION * len(pool)))) if len(pool) else 0
        va_idx.append(pool[:k]); tr_idx.append(pool[k:])
    return np.sort(np.concatenate(tr_idx)), np.sort(np.concatenate(va_idx))

In [ ]:
# ---------------------------------------------------------------------
# Feature sources and scorers. Embeddings are read through mmap and
# converted to float32 one chunk at a time - never as a whole matrix.
# ---------------------------------------------------------------------
class ArrayFeatures:
    def __init__(self, arr):
        self.arr = arr
        self.n, self.d = arr.shape
    def rows(self, idx):
        return np.asarray(self.arr[idx], dtype=np.float32)

class CentroidScorer:
    kind = "cosine"
    def __init__(self, C):
        self.C = C.astype(np.float32)
    def score_block(self, X):
        Xn = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-8)
        return (Xn @ self.C.T).astype(np.float32)

class LinearNumpyScorer:
    kind = "margin"
    def __init__(self, W, b):
        self.W = W.astype(np.float32); self.b = b.astype(np.float32)
    def score_block(self, X):
        return (X @ self.W.T + self.b).astype(np.float32)

class TorchHeadScorer:
    kind = "prob"
    def __init__(self, module, device):
        self.module = module.eval(); self.device = device
    def score_block(self, X):
        with torch.inference_mode():
            t = torch.from_numpy(np.ascontiguousarray(X)).to(self.device)
            return torch.sigmoid(self.module(t)).float().cpu().numpy()

# ---------------------------------------------------------------------
# Metric accumulators. Nothing dense of shape (n_rows, n_labels) is ever
# kept beyond one chunk; the running state is three int64 vectors per level.
# ---------------------------------------------------------------------
_QUANTILES = [0.50, 0.70, 0.80, 0.90, 0.95, 0.98, 0.99, 0.995, 0.998, 0.999, 0.9995, 0.9999]

def _iter_chunks(idx, size):
    for i0 in range(0, len(idx), size):
        yield idx[i0:i0 + size]

def _dense_chunk(Y, idx):
    # chunk-local dense view, at most EVAL_CHUNK rows. The full matrix stays sparse.
    return np.asarray(Y[idx].todense(), dtype=np.uint8)

def tune_thresholds(scorer, feats, Y, maps, idx):
    """Choose one threshold per EC level, on the train-derived validation slice only."""
    sample = {lv: [] for lv in (1, 2, 3, 4)}
    for c in list(_iter_chunks(idx, EVAL_CHUNK))[:2]:
        S = scorer.score_block(feats.rows(c))
        for lv in (1, 2, 3, 4):
            v = apply_level(S, maps[lv]).ravel()
            if v.size > 200000:
                v = v[np.random.RandomState(SEED).choice(v.size, 200000, replace=False)]
            sample[lv].append(v)
        del S
    grids = {lv: np.unique(np.quantile(np.concatenate(sample[lv]), _QUANTILES)) for lv in (1, 2, 3, 4)}
    tp = {lv: np.zeros(len(grids[lv]), np.int64) for lv in (1, 2, 3, 4)}
    fp = {lv: np.zeros(len(grids[lv]), np.int64) for lv in (1, 2, 3, 4)}
    fn = {lv: np.zeros(len(grids[lv]), np.int64) for lv in (1, 2, 3, 4)}
    for c in _iter_chunks(idx, EVAL_CHUNK):
        S = scorer.score_block(feats.rows(c))
        T = _dense_chunk(Y, c)
        for lv in (1, 2, 3, 4):
            Sl = apply_level(S, maps[lv]); Tl = apply_level(T, maps[lv]).astype(bool)
            for gi, thr in enumerate(grids[lv]):
                Pl = Sl >= thr
                tp[lv][gi] += int((Pl & Tl).sum())
                fp[lv][gi] += int((Pl & ~Tl).sum())
                fn[lv][gi] += int((~Pl & Tl).sum())
        del S, T
    best = {}
    for lv in (1, 2, 3, 4):
        f1 = 2 * tp[lv] / np.maximum(2 * tp[lv] + fp[lv] + fn[lv], 1)
        best[lv] = float(grids[lv][int(np.argmax(f1))])
    return best

def evaluate_scores(scorer, feats, Y, maps, thresholds, idx, freq_bins=None):
    """Chunked evaluation with per-label TP/FP/FN accumulators at all four EC levels."""
    n_lv = {lv: len(maps[lv]["names"]) for lv in (1, 2, 3, 4)}
    tp = {lv: np.zeros(n_lv[lv], np.int64) for lv in (1, 2, 3, 4)}
    fp = {lv: np.zeros(n_lv[lv], np.int64) for lv in (1, 2, 3, 4)}
    fn = {lv: np.zeros(n_lv[lv], np.int64) for lv in (1, 2, 3, 4)}
    subset = {lv: 0 for lv in (1, 2, 3, 4)}
    jacc = {lv: 0.0 for lv in (1, 2, 3, 4)}
    top1 = {lv: 0 for lv in (1, 2, 3, 4)}
    total = 0
    for c in _iter_chunks(idx, EVAL_CHUNK):
        S = scorer.score_block(feats.rows(c))
        T = _dense_chunk(Y, c)
        total += len(c)
        for lv in (1, 2, 3, 4):
            m = maps[lv]
            Sl = apply_level(S, m); Tl = apply_level(T, m).astype(bool)
            Pl = Sl >= thresholds[lv]
            empty = ~Pl.any(axis=1)
            if empty.any():
                Pl[np.where(empty)[0], Sl[empty].argmax(axis=1)] = True
            tp[lv] += (Pl & Tl).sum(axis=0)
            fp[lv] += (Pl & ~Tl).sum(axis=0)
            fn[lv] += (~Pl & Tl).sum(axis=0)
            inter = (Pl & Tl).sum(axis=1); union = (Pl | Tl).sum(axis=1)
            jacc[lv] += float((inter / np.maximum(union, 1)).sum())
            subset[lv] += int((Pl == Tl).all(axis=1).sum())
            a = Sl.argmax(axis=1)
            top1[lv] += int(Tl[np.arange(len(c)), a].sum())
        del S, T
    return _finalise(tp, fp, fn, subset, jacc, top1, total, freq_bins)

def evaluate_from_sets(pred_sets, Y, maps, idx, freq_bins=None):
    """Same metrics for a method that emits label sets directly (Stage 11, DIAMOND)."""
    n_lv = {lv: len(maps[lv]["names"]) for lv in (1, 2, 3, 4)}
    tp = {lv: np.zeros(n_lv[lv], np.int64) for lv in (1, 2, 3, 4)}
    fp = {lv: np.zeros(n_lv[lv], np.int64) for lv in (1, 2, 3, 4)}
    fn = {lv: np.zeros(n_lv[lv], np.int64) for lv in (1, 2, 3, 4)}
    subset = {lv: 0 for lv in (1, 2, 3, 4)}
    jacc = {lv: 0.0 for lv in (1, 2, 3, 4)}
    top1 = {lv: 0 for lv in (1, 2, 3, 4)}
    total = 0
    L = Y.shape[1]
    for c in _iter_chunks(idx, EVAL_CHUNK):
        Pd = np.zeros((len(c), L), dtype=np.uint8)   # chunk-local dense only
        for r, row in enumerate(c):
            for j in pred_sets.get(int(row), ()):    # j is a level-4 label index
                Pd[r, j] = 1
        T = _dense_chunk(Y, c)
        total += len(c)
        for lv in (1, 2, 3, 4):
            m = maps[lv]
            Pl = apply_level(Pd, m).astype(bool); Tl = apply_level(T, m).astype(bool)
            tp[lv] += (Pl & Tl).sum(axis=0)
            fp[lv] += (Pl & ~Tl).sum(axis=0)
            fn[lv] += (~Pl & Tl).sum(axis=0)
            inter = (Pl & Tl).sum(axis=1); union = (Pl | Tl).sum(axis=1)
            jacc[lv] += float((inter / np.maximum(union, 1)).sum())
            subset[lv] += int((Pl == Tl).all(axis=1).sum())
            hit = (Pl & Tl).any(axis=1)
            top1[lv] += int(hit.sum())
        del Pd, T
    return _finalise(tp, fp, fn, subset, jacc, top1, total, freq_bins)

def _prf(tp, fp, fn):
    p = tp / np.maximum(tp + fp, 1)
    r = tp / np.maximum(tp + fn, 1)
    f = 2 * p * r / np.maximum(p + r, 1e-12)
    return p, r, f

def _finalise(tp, fp, fn, subset, jacc, top1, total, freq_bins):
    out = {}
    for lv in (1, 2, 3, 4):
        TP, FP, FN = tp[lv].sum(), fp[lv].sum(), fn[lv].sum()
        micro_p = TP / max(TP + FP, 1); micro_r = TP / max(TP + FN, 1)
        micro_f = 2 * micro_p * micro_r / max(micro_p + micro_r, 1e-12)
        p, r, f = _prf(tp[lv].astype(float), fp[lv].astype(float), fn[lv].astype(float))
        present = (tp[lv] + fn[lv]) > 0
        d = {"micro_precision": float(micro_p), "micro_recall": float(micro_r),
             "micro_f1": float(micro_f),
             "macro_precision": float(p[present].mean()) if present.any() else 0.0,
             "macro_recall": float(r[present].mean()) if present.any() else 0.0,
             "macro_f1": float(f[present].mean()) if present.any() else 0.0,
             "subset_accuracy": subset[lv] / max(total, 1),
             "jaccard": jacc[lv] / max(total, 1),
             "top1_accuracy": top1[lv] / max(total, 1),
             "labels_in_test": int(present.sum())}
        if freq_bins is not None:
            bins = {}
            for b, (name, _, _) in enumerate(BIN_EDGES):
                sel = present & (freq_bins[lv] == b)
                bins[name] = {"macro_f1": float(f[sel].mean()) if sel.any() else None,
                              "n_labels": int(sel.sum())}
            d["frequency_bins"] = bins
        out["level{}".format(lv)] = d
    out["n_rows"] = total
    return out

METRIC_ROWS = []

def record_metrics(split, head, variant, encoder, metrics):
    for lv in (1, 2, 3, 4):
        d = metrics["level{}".format(lv)]
        for k, v in d.items():
            if k == "frequency_bins":
                for bname, bd in v.items():
                    if bd["macro_f1"] is not None:
                        METRIC_ROWS.append({"split": split, "head": head, "encoder": encoder,
                                            "test_variant": variant, "level": lv,
                                            "metric": "macro_f1[{}]".format(bname),
                                            "value": bd["macro_f1"], "n_labels": bd["n_labels"]})
            else:
                METRIC_ROWS.append({"split": split, "head": head, "encoder": encoder,
                                    "test_variant": variant, "level": lv,
                                    "metric": k, "value": v, "n_labels": d["labels_in_test"]})

def print_metrics(title, metrics):
    print("\n" + title)
    rows = []
    for lv in (1, 2, 3, 4):
        d = metrics["level{}".format(lv)]
        rows.append({"level": lv, "micro_F1": round(d["micro_f1"], 4),
                     "macro_F1": round(d["macro_f1"], 4),
                     "subset_acc": round(d["subset_accuracy"], 4),
                     "jaccard": round(d["jaccard"], 4),
                     "top1_acc": round(d["top1_accuracy"], 4),
                     "labels": d["labels_in_test"]})
    print(pd.DataFrame(rows).to_string(index=False))
    if "frequency_bins" in metrics["level4"]:
        fb = metrics["level4"]["frequency_bins"]
        print("  level-4 macro-F1 by training frequency: " +
              "  ".join("{}={}".format(k, "n/a" if v["macro_f1"] is None else round(v["macro_f1"], 4))
                        for k, v in fb.items()))

def dedup_index(split, n_test):
    """Row indices of the 'dedup' test variant: official rows minus verbatim-leaked rows."""
    p = res_path("dedup_mask_split{}_fold{}.npy".format(split, FOLD))
    if not p.exists():
        return np.arange(n_test)
    mask = np.load(p)
    if len(mask) != n_test:      # smoke subsets change the row count
        return np.arange(n_test)
    return np.where(mask)[0]

In [ ]:
# Free the raw split frames held for the audit; later stages reload what they need.
_SPLIT_CACHE.clear()
gc.collect()
print("split cache cleared")

## Stage 5 — Honest amino-acid-composition baseline

The same 20-dimensional composition features as the legacy work, but trained on the official training split and evaluated on the official held-out test split, as a proper multi-label problem.
This is the row that says what classical features achieve when they are evaluated honestly. Peak RAM ≈ 2 GB.

In [ ]:
from sklearn.multiclass import OneVsRestClassifier

def _ovr_to_linear(ovr, n_labels, d):
    W = np.zeros((n_labels, d), dtype=np.float32)
    b = np.full(n_labels, -1e6, dtype=np.float32)
    for k, est in zip(ovr.classes_, ovr.estimators_):
        if hasattr(est, "coef_"):
            W[int(k)] = np.asarray(est.coef_).ravel()
            b[int(k)] = float(np.asarray(est.intercept_).ravel()[0])
    return LinearNumpyScorer(W, b)

def run_aac_baseline(split):
    space = get_label_space(split)
    tr = load_split(split, FOLD, "train"); te = load_split(split, FOLD, "test")
    Xtr = ArrayFeatures(aac_matrix(tr["seq"].tolist()))
    Xte = ArrayFeatures(aac_matrix(te["seq"].tolist()))
    fit_idx, val_idx = train_val_indices(space)

    print("  fitting {} one-vs-rest LinearSVC problems on {} x 20 ...".format(
        space["n_labels"], len(fit_idx)))
    ovr = OneVsRestClassifier(LinearSVC(random_state=SEED, dual="auto", max_iter=2000, tol=1e-3),
                              n_jobs=-1)
    ovr.fit(Xtr.rows(fit_idx), space["Ytr"][fit_idx])
    scorer = _ovr_to_linear(ovr, space["n_labels"], 20)
    del ovr; gc.collect()

    thr = tune_thresholds(scorer, Xtr, space["Ytr"], space["maps"], val_idx)
    out = {"thresholds": thr, "variants": {}}
    official = np.arange(space["n_test"])
    out["variants"]["official"] = evaluate_scores(scorer, Xte, space["Yte"], space["maps"],
                                                  thr, official, space["bins"])
    ded = dedup_index(split, space["n_test"])
    out["n_leaked_test_rows"] = int(space["n_test"] - len(ded))
    if out["n_leaked_test_rows"] > 0:
        out["variants"]["dedup"] = evaluate_scores(scorer, Xte, space["Yte"], space["maps"],
                                                   thr, ded, space["bins"])
    del Xtr, Xte, scorer; gc.collect()
    return out

if RUN_AAC_BASELINE:
    with stage_timer("Stage 5 - honest AAC baseline"):
        AAC_RESULTS = {}
        for s in SPLITS_TO_RUN:
            print("\nsplit{}".format(s))
            r = cached_json("stage5_aac_split{}_fold{}.json".format(s, FOLD),
                            lambda s=s: run_aac_baseline(s))
            AAC_RESULTS[s] = r
            print_metrics("AAC + LinearSVC | split{} | official".format(s), r["variants"]["official"])
            if "dedup" in r["variants"]:
                print_metrics("AAC + LinearSVC | split{} | dedup".format(s), r["variants"]["dedup"])
            else:
                print("  dedup == official (0 leaked rows)")
            record_metrics(s, "H0_AAC_LinearSVC", "official", "aac20", r["variants"]["official"])
            if "dedup" in r["variants"]:
                record_metrics(s, "H0_AAC_LinearSVC", "dedup", "aac20", r["variants"]["dedup"])
else:
    AAC_RESULTS = {}
    print("Stage 5 skipped (RUN_AAC_BASELINE = False)")

**What this supports.** The composition baseline is evaluated on the same supplied test rows as the embedding heads. Their representations and classifier types differ. The learned heads and composition classifier use the train-derived fit/validation partition, while DIAMOND searches the complete supplied training set. The historical 0.27778 accuracy used a different, invalid test set and is excluded from the main comparison.


## Stage 6 — ESM-2 embedding generation with checkpoint and resume

The encoder is frozen and every sequence is embedded exactly once; the result is cached to persistent project storage, which is the permanent fix for the project's repeated re-encoding cost.
Work is written in shards of `SHARD_SIZE` sequences, so a Colab disconnect costs at most one shard. Peak RAM ≈ 3 GB, VRAM ≈ 2–4 GB.

In [ ]:
_ENCODERS = {}

def get_encoder(model_id):
    """Load a frozen ESM-2 encoder in fp16 on the GPU. Loaded before any sequence data."""
    if model_id in _ENCODERS:
        return _ENCODERS[model_id]
    from transformers import AutoTokenizer, AutoModel
    tok = AutoTokenizer.from_pretrained(model_id)
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    try:
        # Direct half-precision load: no transient fp32 copy is ever materialised.
        model = AutoModel.from_pretrained(model_id, dtype=dtype)
    except TypeError:
        try:
            model = AutoModel.from_pretrained(model_id, torch_dtype=dtype)
        except TypeError:
            model = AutoModel.from_pretrained(model_id)
            if DEVICE == "cuda":
                model = model.half()          # in-place; the fp32 storage is released here
    model = model.to(DEVICE).eval()
    for p_ in model.parameters():
        p_.requires_grad_(False)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    d = int(model.config.hidden_size)
    print("encoder {} loaded | hidden={} | dtype={} | device={}".format(model_id, d, dtype, DEVICE))
    _ENCODERS[model_id] = (tok, model, d)
    return _ENCODERS[model_id]

def check_alphabet(tok, seqs):
    """Every residue character present in the data must exist in the tokenizer vocabulary."""
    vocab = set(tok.get_vocab().keys())
    missing = sorted(set("".join(seqs)) - vocab)   # one pass over the whole corpus
    if missing:
        print("  WARNING: characters absent from the tokenizer vocabulary: {} -> mapped to X".format(missing))
    return missing

def _sanitise(seqs, missing):
    if not missing:
        return seqs
    table = str.maketrans({m: "X" for m in missing})
    return [s.translate(table) for s in seqs]

def _token_budget_batches(lengths, budget):
    batches, cur, curmax = [], [], 0
    for i, L in enumerate(lengths):
        m = max(curmax, L)
        if cur and (len(cur) + 1) * m > budget:
            batches.append(cur); cur = [i]; curmax = L
        else:
            cur.append(i); curmax = m
    if cur:
        batches.append(cur)
    return batches

In [ ]:
CUR_TOKEN_BUDGET = MAX_TOKENS_BATCH
_FIRST_BATCH_CHECKED = False

def _forward(sub_seqs, tok, model):
    """Mean-pool over real residue tokens only: BOS, EOS and padding are all excluded."""
    global _FIRST_BATCH_CHECKED
    enc = tok(sub_seqs, return_tensors="pt", padding=True,
              truncation=True, max_length=MAX_LEN + 2)     # +2 for the BOS and EOS tokens
    enc = {k: v.to(DEVICE) for k, v in enc.items() if k in ("input_ids", "attention_mask")}
    am = enc["attention_mask"]
    mask = am.clone()
    mask[:, 0] = 0                                    # drop BOS
    last = am.sum(dim=1) - 1                          # index of EOS for each row
    mask[torch.arange(mask.size(0), device=am.device), last] = 0
    if not _FIRST_BATCH_CHECKED:
        want = torch.tensor([min(len(s), MAX_LEN) for s in sub_seqs], device=am.device)
        got = mask.sum(dim=1)
        assert torch.equal(got, want), \
            "residue-token mask is wrong: got {} expected {}".format(got[:5].tolist(), want[:5].tolist())
        print("  residue-mask check OK (BOS/EOS/padding excluded, counts match the sequences)")
        _FIRST_BATCH_CHECKED = True
    with torch.inference_mode():
        # pool in float32: a float16 sum over up to 1,022 tokens can overflow
        hs = model(**enc).last_hidden_state.float()
        w = mask.float().unsqueeze(-1)
        emb = (hs * w).sum(dim=1) / w.sum(dim=1).clamp(min=1.0)
        out = emb.cpu().numpy().astype(np.float16)
    del enc, am, mask, hs, w, emb
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return out

def _embed_list(seqs, tok, model, d):
    """Length-sorted, token-budgeted embedding of one shard, halving the budget on CUDA OOM."""
    global CUR_TOKEN_BUDGET
    out = np.zeros((len(seqs), d), dtype=np.float16)
    lens = [min(len(s), MAX_LEN) + 2 for s in seqs]

    def run(indices):
        global CUR_TOKEN_BUDGET
        try:
            out[indices] = _forward([seqs[i] for i in indices], tok, model)
        except Exception as e:
            oom = isinstance(e, getattr(torch.cuda, "OutOfMemoryError", tuple())) or \
                  ("out of memory" in str(e).lower())
            if not oom or len(indices) == 1:
                raise
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            CUR_TOKEN_BUDGET = max(1024, CUR_TOKEN_BUDGET // 2)
            print("  CUDA OOM -> token budget halved to {}".format(CUR_TOKEN_BUDGET))
            mid = len(indices) // 2
            run(indices[:mid]); run(indices[mid:])

    for bt in _token_budget_batches(lens, CUR_TOKEN_BUDGET):
        run(bt)
    return out

def embed_part(split, part, model_id, tag):
    """Embed one (split, part) with shard checkpointing. Returns (n_sequences, n_residues, seconds)."""
    ep, ip = emb_path(split, FOLD, part, tag), ids_path(split, FOLD, part, tag)
    df = load_split(split, FOLD, part)
    seqs, ids = df["seq"].tolist(), df["id"].tolist()
    n_res = int(sum(len(s) for s in seqs))
    if ep.exists() and ip.exists() and not FORCE_RECOMPUTE:
        print("  cached -> {}".format(ep.name))
        return len(seqs), n_res, 0.0

    tok, model, d = get_encoder(model_id)
    seqs = _sanitise(seqs, check_alphabet(tok, seqs))
    n = len(seqs)
    shard_dir = ep.parent / (ep.stem + "_shards")
    shard_dir.mkdir(parents=True, exist_ok=True)
    order_file = shard_dir / "order.npy"
    if order_file.exists():
        order = np.load(order_file)
        assert len(order) == n, "saved shard order does not match the data - delete {}".format(shard_dir)
    else:
        order = np.argsort(np.array([len(s) for s in seqs]), kind="stable")
        np.save(order_file, order)
    prog_file = shard_dir / "progress.json"

    n_shards = int(math.ceil(n / SHARD_SIZE))
    t0 = time.time()
    for si in range(n_shards):
        sf = shard_dir / "shard{:05d}.npy".format(si)
        idx = order[si * SHARD_SIZE:(si + 1) * SHARD_SIZE]
        if sf.exists():
            try:
                if np.load(sf, mmap_mode="r").shape == (len(idx), d):
                    continue
            except Exception:
                pass
            sf.unlink()
        arr = _embed_list([seqs[i] for i in idx], tok, model, d)
        tmp = sf.with_suffix(".tmp.npy")
        np.save(tmp, arr)
        tmp.replace(sf)
        (shard_dir / "shard{:05d}.ids.txt".format(si)).write_text("\n".join(ids[i] for i in idx))
        prog_file.write_text(json.dumps({"n": n, "d": d, "shard_size": SHARD_SIZE,
                                         "done": si + 1, "total": n_shards}))
        del arr
        if si % 10 == 0 or si == n_shards - 1:
            print("    shard {}/{}  ({:.1f}s elapsed)".format(si + 1, n_shards, time.time() - t0))
    secs = time.time() - t0

    # Assemble back into ORIGINAL row order through a memmap: peak RAM is one shard.
    tmp_final = ep.with_suffix(".tmp.npy")
    final = np.lib.format.open_memmap(tmp_final, mode="w+", dtype=np.float16, shape=(n, d))
    for si in range(n_shards):
        idx = order[si * SHARD_SIZE:(si + 1) * SHARD_SIZE]
        final[idx] = np.load(shard_dir / "shard{:05d}.npy".format(si))
    final.flush()
    del final; gc.collect()
    tmp_final.replace(ep)
    ip.write_text("\n".join(ids))
    print("  wrote {}  shape=({}, {})  {:.1f}s".format(ep.name, n, d, secs))
    return n, n_res, secs

In [ ]:
def run_embedding_stage(model_id, tag):
    jobs = []
    est = {(r["split"], "train"): r["train_residues"] for r in STAGE3["per_split"]}
    est.update({(r["split"], "test"): r["test_residues"] for r in STAGE3["per_split"]})
    for s in EMBED_SPLITS:
        for part in ("test", "train"):
            jobs.append((s, part, est.get((s, part), 0)))
    jobs.sort(key=lambda j: j[2])          # smallest first -> throughput probe before the long work
    measured = None
    for k, (s, part, res_est) in enumerate(jobs):
        print("\n[{}] split{} {}".format(tag, s, part))
        n, n_res, secs = embed_part(s, part, model_id, tag)
        if secs > 5 and measured is None:
            measured = (n / secs, n_res / secs)
            print("\n  measured throughput: {:.1f} sequences/s, {:.0f} residues/s".format(*measured))
            rem = [(a, b, c) for (a, b, c) in jobs[k + 1:]]
            if rem:
                print("  extrapolated ETA for the remaining work:")
                tot = 0.0
                for (a, b, c) in rem:
                    e = c / measured[1] / 60.0
                    tot += e
                    print("    split{:<4} {:<6} ~{:6.1f} min".format(a, b, e))
                print("    {:<12} ~{:6.1f} min total".format("ALL", tot))

if EMBED_SPLITS:
    with stage_timer("Stage 6 - ESM-2 embedding generation"):
        for _mid, _tag in MODEL_PASSES:
            run_embedding_stage(_mid, _tag)
else:
    print("Stage 6 skipped: no splits selected")

## Stage 7 — Embedding sanity checks

Every cached matrix is checked for shape, dtype, missing values, all-zero rows and identifier alignment before any model is trained on it.
Shards are removed only after these checks pass, so a failed check leaves the resumable state intact.

In [ ]:
def check_embeddings(split, part, tag):
    ep, ip = emb_path(split, FOLD, part, tag), ids_path(split, FOLD, part, tag)
    df = load_split(split, FOLD, part)
    A = np.load(ep, mmap_mode="r")
    assert A.dtype == np.float16, "{}: dtype is {}, expected float16".format(ep.name, A.dtype)
    assert A.shape[0] == len(df), "{}: {} rows, expected {}".format(ep.name, A.shape[0], len(df))
    n_nan = n_zero = 0
    norms = []
    for i0 in range(0, A.shape[0], EVAL_CHUNK):
        blk = np.asarray(A[i0:i0 + EVAL_CHUNK], dtype=np.float32)
        n_nan += int((~np.isfinite(blk)).sum())
        nb = np.linalg.norm(blk, axis=1)
        n_zero += int((nb == 0).sum())
        norms.append(nb)
    assert n_nan == 0, "{}: {} non-finite values".format(ep.name, n_nan)
    assert n_zero == 0, "{}: {} all-zero rows".format(ep.name, n_zero)
    ids = ip.read_text().split("\n")
    assert len(ids) == len(df), "{}: {} ids, expected {}".format(ip.name, len(ids), len(df))
    assert ids == df["id"].tolist(), "{}: id order does not match the source dataframe".format(ip.name)
    mean_norm = float(np.concatenate(norms).mean())
    del A; gc.collect()
    return {"split": split, "part": part, "encoder": tag, "rows": len(df),
            "dim": int(np.load(ep, mmap_mode="r").shape[1]),
            "mean_l2_norm": round(mean_norm, 3), "nan": n_nan, "zero_rows": n_zero}

if EMBED_SPLITS:
    with stage_timer("Stage 7 - embedding sanity checks"):
        rows = []
        for _mid, _tag in MODEL_PASSES:
            for s in EMBED_SPLITS:
                for part in ("train", "test"):
                    rows.append(check_embeddings(s, part, _tag))
                    if not KEEP_SHARDS:
                        sd = emb_path(s, FOLD, part, _tag).parent / (
                            emb_path(s, FOLD, part, _tag).stem + "_shards")
                        if sd.is_dir():
                            shutil.rmtree(sd, ignore_errors=True)
        print(pd.DataFrame(rows).to_string(index=False))
        print("Stage 7 OK : every cached embedding matrix passed all checks"
              + ("" if KEEP_SHARDS else "; shard checkpoints removed"))

## Stage 9 — Prediction heads

Three heads use the same frozen embeddings: a cosine nearest-EC-centroid rule, a linear classifier, and a small neural network. The centroid rule uses class averages without gradient training. It provides a simple control, but does not reproduce CLEAN's learned contrastive projection or its complete inference procedure.


In [ ]:
import torch.nn as nn

def open_features(split, part, tag):
    return ArrayFeatures(np.load(emb_path(split, FOLD, part, tag), mmap_mode="r"))

def build_centroids(space, feats, fit_idx):
    L, D = space["n_labels"], feats.d
    C = np.zeros((L, D), dtype=np.float32)
    cnt = np.zeros(L, dtype=np.float32)
    Y = space["Ytr"]
    for c in _iter_chunks(fit_idx, EVAL_CHUNK):
        X = feats.rows(c)
        X /= np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-8)
        Ys = Y[c].astype(np.float32)
        C += np.asarray(Ys.T @ X, dtype=np.float32)
        cnt += np.asarray(Ys.sum(axis=0)).ravel()
        del X, Ys
    C /= np.maximum(cnt, 1.0)[:, None]
    C /= np.maximum(np.linalg.norm(C, axis=1, keepdims=True), 1e-8)
    empty = int((cnt == 0).sum())
    if empty:
        print("  note: {} of {} labels have no training row in the fit slice; "
              "their centroid is the zero vector".format(empty, L))
    return CentroidScorer(C)

def make_module(kind, D, L):
    if kind == "H2_linear":
        return nn.Linear(D, L)
    return nn.Sequential(nn.Linear(D, HEAD_HIDDEN), nn.GELU(),
                         nn.Dropout(HEAD_DROPOUT), nn.Linear(HEAD_HIDDEN, L))

def train_torch_head(kind, space, feats, fit_idx, val_idx, ckpt):
    D, L = feats.d, space["n_labels"]
    module = make_module(kind, D, L).to(DEVICE)
    if ckpt.exists() and not FORCE_RECOMPUTE:
        module.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        print("  cached head -> {}".format(ckpt.name))
        return module
    opt = torch.optim.AdamW(module.parameters(), lr=HEAD_LR, weight_decay=HEAD_WEIGHT_DECAY)
    crit = nn.BCEWithLogitsLoss()
    Y = space["Ytr"]
    rng = np.random.RandomState(SEED)
    best, bad, best_state = float("inf"), 0, None
    for ep in range(EFFECTIVE_EPOCHS):
        module.train()
        order = rng.permutation(len(fit_idx))
        tot = 0.0
        for i0 in range(0, len(order), HEAD_BATCH):
            b = np.sort(fit_idx[order[i0:i0 + HEAD_BATCH]])
            x = torch.from_numpy(feats.rows(b)).to(DEVICE)
            y = torch.from_numpy(_dense_chunk(Y, b).astype(np.float32)).to(DEVICE)
            opt.zero_grad()
            loss = crit(module(x), y)
            loss.backward(); opt.step()
            tot += float(loss) * len(b)
            del x, y
        module.eval()
        vtot = 0.0
        with torch.inference_mode():
            for b in _iter_chunks(val_idx, HEAD_BATCH):
                x = torch.from_numpy(feats.rows(b)).to(DEVICE)
                y = torch.from_numpy(_dense_chunk(Y, b).astype(np.float32)).to(DEVICE)
                vtot += float(crit(module(x), y)) * len(b)
                del x, y
        vloss = vtot / max(len(val_idx), 1)
        if ep % 5 == 0 or ep == EFFECTIVE_EPOCHS - 1:
            print("    epoch {:>3}  train {:.5f}  val {:.5f}".format(ep, tot / max(len(fit_idx), 1), vloss))
        if vloss < best - 1e-5:
            best, bad = vloss, 0
            best_state = {k: v.detach().cpu().clone() for k, v in module.state_dict().items()}
        else:
            bad += 1
            if bad >= EARLY_STOP_PATIENCE:
                print("    early stop at epoch {} (best val {:.5f})".format(ep, best))
                break
    if best_state is not None:
        module.load_state_dict(best_state)
    torch.save(module.state_dict(), ckpt)
    return module

HEADS = {}

def build_heads(split, tag):
    space = get_label_space(split)
    feats = open_features(split, "train", tag)
    fit_idx, val_idx = train_val_indices(space)
    made = {}

    cfile = res_path("head_H1_centroid_{}_split{}_fold{}.npy".format(tag, split, FOLD))
    if cfile.exists() and not FORCE_RECOMPUTE:
        sc = CentroidScorer(np.load(cfile))
        print("  cached head -> {}".format(cfile.name))
    else:
        sc = build_centroids(space, feats, fit_idx)
        np.save(cfile, sc.C)
    made["H1_centroid"] = sc

    for kind in ("H2_linear", "H3_mlp"):
        ck = res_path("head_{}_{}_split{}_fold{}.pt".format(kind, tag, split, FOLD))
        print("  {}".format(kind))
        made[kind] = TorchHeadScorer(train_torch_head(kind, space, feats, fit_idx, val_idx, ck), DEVICE)

    thr_file = res_path("thresholds_{}_split{}_fold{}.json".format(tag, split, FOLD))
    if thr_file.exists() and not FORCE_RECOMPUTE:
        thr = {k: {int(a): b for a, b in v.items()} for k, v in json.loads(thr_file.read_text()).items()}
    else:
        thr = {name: tune_thresholds(sc_, feats, space["Ytr"], space["maps"], val_idx)
               for name, sc_ in made.items()}
        thr_file.write_text(json.dumps(thr, indent=1))
    for name, sc_ in made.items():
        HEADS[(tag, split, name)] = (sc_, thr[name])
    del feats; gc.collect()

if EMBED_SPLITS:
    with stage_timer("Stage 9 - head training"):
        for _mid, _tag in MODEL_PASSES:
            for s in EMBED_SPLITS:
                print("\n[{}] split{}".format(_tag, s))
                build_heads(s, _tag)

## Stage 10 — Evaluation on the supplied held-out test sets

Test predictions are scored in chunks with per-label true-positive, false-positive and false-negative counters. Only chunk-sized dense matrices are created. Top-1 accuracy checks the highest-scoring label, while multi-label metrics use validation-selected thresholds and a top-1 fallback. The audit and earlier setup stages also read test labels; thresholds and head fitting use only the supplied training partition.


In [ ]:
def evaluate_head(split, tag, name):
    space = get_label_space(split)
    scorer, thr = HEADS[(tag, split, name)]
    thr = {int(k): float(v) for k, v in thr.items()}
    feats = open_features(split, "test", tag)
    out = {"thresholds": thr, "variants": {}}
    out["variants"]["official"] = evaluate_scores(scorer, feats, space["Yte"], space["maps"],
                                                  thr, np.arange(space["n_test"]), space["bins"])
    ded = dedup_index(split, space["n_test"])
    out["n_leaked_test_rows"] = int(space["n_test"] - len(ded))
    if out["n_leaked_test_rows"] > 0:
        out["variants"]["dedup"] = evaluate_scores(scorer, feats, space["Yte"], space["maps"],
                                                   thr, ded, space["bins"])
    del feats; gc.collect()
    return out

EMB_RESULTS = {}
if EMBED_SPLITS:
    with stage_timer("Stage 10 - evaluation"):
        for _mid, _tag in MODEL_PASSES:
            for s in EMBED_SPLITS:
                for name in ("H1_centroid", "H2_linear", "H3_mlp"):
                    r = cached_json("stage10_{}_{}_split{}_fold{}.json".format(name, _tag, s, FOLD),
                                    lambda s=s, t=_tag, n=name: evaluate_head(s, t, n))
                    EMB_RESULTS[(_tag, s, name)] = r
                    print_metrics("{} | {} | split{} | official".format(name, _tag, s),
                                  r["variants"]["official"])
                    record_metrics(s, name, "official", _tag, r["variants"]["official"])
                    if "dedup" in r["variants"]:
                        print_metrics("{} | {} | split{} | dedup".format(name, _tag, s),
                                      r["variants"]["dedup"])
                        record_metrics(s, name, "dedup", _tag, r["variants"]["dedup"])
                    else:
                        print("  dedup == official (0 leaked rows)")

**What these tables support.** They compare representations and heads on the same supplied test rows, at all four EC levels and across training-frequency bins. DIAMOND uses the complete supplied training set as its reference; fitted classifiers and centroids use the fit slice after reserving validation rows. This is not a controlled representation-only comparison or a numerical head-to-head with published methods evaluated on different test data.


## Stage 11 — DIAMOND homology-transfer baseline

Sequence-similarity transfer is the field's real reference point, so it belongs in the comparison ladder.
If DIAMOND cannot be installed the stage prints a warning and is skipped; it never stops the notebook.

In [ ]:
def ensure_diamond():
    try:
        r = subprocess.run(["diamond", "--version"], capture_output=True, text=True, timeout=60)
        if r.returncode == 0:
            print("diamond already available:", r.stdout.strip())
            return True
    except Exception:
        pass
    try:
        subprocess.run(["apt-get", "install", "-y", "-qq", "diamond-aligner"],
                       capture_output=True, text=True, timeout=600)
        r = subprocess.run(["diamond", "--version"], capture_output=True, text=True, timeout=60)
        if r.returncode == 0:
            print("diamond installed:", r.stdout.strip())
            return True
        print("WARNING: diamond installed but not runnable -", r.stderr.strip()[:300])
    except Exception as e:
        print("WARNING: diamond installation failed -", str(e)[:300])
    print("Stage 11 will be skipped. This does not affect any other stage.")
    return False

def _write_fasta(path, ids, seqs):
    with open(path, "w") as f:
        for i, s in zip(ids, seqs):
            f.write(">{}\n{}\n".format(i, s))

def run_diamond_baseline(split):
    space = get_label_space(split)
    tr = load_split(split, FOLD, "train"); te = load_split(split, FOLD, "test")
    work = Path(tempfile.mkdtemp(prefix="diamond_"))
    db_fa, q_fa, out_tsv = work / "train.fasta", work / "test.fasta", work / "hits.tsv"
    _write_fasta(db_fa, ["{}_{}".format(k, i) for k, i in enumerate(tr["id"])], tr["seq"])
    _write_fasta(q_fa, ["{}_{}".format(k, i) for k, i in enumerate(te["id"])], te["seq"])
    subprocess.run(["diamond", "makedb", "--in", str(db_fa), "-d", str(work / "db"), "--quiet"],
                   check=True)
    subprocess.run(["diamond", "blastp", "-q", str(q_fa), "-d", str(work / "db"),
                    "-o", str(out_tsv), "-k", "1", "--quiet", "--outfmt", "6",
                    "qseqid", "sseqid", "bitscore"], check=True)
    shutil.copy(out_tsv, res_path("diamond_hits_split{}_fold{}.tsv".format(split, FOLD)))

    train_label_idx = [sorted({space["index"][e] for e in parse_ec_set(s) if e in space["index"]})
                       for s in tr["ec"]]
    pred = {}
    with open(out_tsv) as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 2:
                continue
            qi = int(parts[0].split("_", 1)[0]); si = int(parts[1].split("_", 1)[0])
            if qi not in pred:
                pred[qi] = train_label_idx[si]
    n_hit = len(pred)
    m = evaluate_from_sets(pred, space["Yte"], space["maps"], np.arange(space["n_test"]), space["bins"])
    out = {"variants": {"official": m}, "test_rows_with_a_hit": n_hit,
           "hit_rate": round(n_hit / max(space["n_test"], 1), 4)}
    ded = dedup_index(split, space["n_test"])
    out["n_leaked_test_rows"] = int(space["n_test"] - len(ded))
    if out["n_leaked_test_rows"] > 0:
        out["variants"]["dedup"] = evaluate_from_sets(pred, space["Yte"], space["maps"], ded, space["bins"])
    shutil.rmtree(work, ignore_errors=True)
    return out

DIAMOND_RESULTS = {}
if RUN_DIAMOND and ensure_diamond():
    with stage_timer("Stage 11 - DIAMOND baseline"):
        for s in SPLITS_TO_RUN:
            print("\nsplit{}".format(s))
            r = cached_json("stage11_diamond_split{}_fold{}.json".format(s, FOLD),
                            lambda s=s: run_diamond_baseline(s))
            DIAMOND_RESULTS[s] = r
            print("  test rows with a hit: {} ({:.1%})".format(r["test_rows_with_a_hit"], r["hit_rate"]))
            print_metrics("DIAMOND top-hit transfer | split{} | official".format(s),
                          r["variants"]["official"])
            record_metrics(s, "B1_DIAMOND", "official", "alignment", r["variants"]["official"])
            if "dedup" in r["variants"]:
                print_metrics("DIAMOND top-hit transfer | split{} | dedup".format(s),
                              r["variants"]["dedup"])
                record_metrics(s, "B1_DIAMOND", "dedup", "alignment", r["variants"]["dedup"])
            else:
                print("  dedup == official (0 leaked rows)")
elif RUN_DIAMOND:
    print("Stage 11 skipped: DIAMOND is unavailable in this environment.")
else:
    print("Stage 11 skipped (RUN_DIAMOND = False)")

**Note on the `top1` column for this row.** DIAMOND transfers the whole EC set of its single best hit, so its `top1_accuracy` is read as "the transferred set contains at least one correct EC", which is a slightly easier question than the single-label `top1` of the scoring heads.

## Stage 13 — Results tables, figures and run manifest

All metrics are written as one tidy long-format CSV, and five figures summarise the run.
The manifest records library versions, hardware, seeds, per-stage wall-clock and a SHA-256 for every input file actually read.

In [ ]:
with stage_timer("Stage 13 - tables and figures"):
    METRICS = pd.DataFrame(METRIC_ROWS, columns=["split", "head", "encoder", "test_variant",
                                                 "level", "metric", "value", "n_labels"])
    mcsv = res_path("metrics_long_fold{}.csv".format(FOLD))
    METRICS.to_csv(mcsv, index=False)
    print("metrics written ->", mcsv.name, "|", len(METRICS), "rows")

    headline = METRICS[(METRICS.metric == "micro_f1") & (METRICS.test_variant == "official")]
    if len(headline):
        piv = headline.pivot_table(index=["head", "encoder"], columns=["level", "split"], values="value")
        print("\nmicro-F1, official test sets:")
        print(piv.round(4).to_string())

In [ ]:
# Colour-blind-safe categorical order (Okabe-Ito). Hues are assigned in a fixed
# order and never cycled; sequential magnitude uses a single hue.
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9"]
GRID = dict(color="#D9D9D9", linewidth=0.8)
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#666666", "font.size": 9})

def _style(ax, title, xlabel, ylabel):
    ax.set_title(title, fontsize=10, loc="left")
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.grid(axis="y", **GRID); ax.set_axisbelow(True)

FIGS = []

# Figure 1 - class-size distribution (log-log), largest split actually modelled.
if EMBED_SPLITS:
    s0 = max(EMBED_SPLITS)
    freq = np.sort(get_label_space(s0)["freq"][4])[::-1]
    fig, ax = plt.subplots(figsize=(5.2, 3.4))
    ax.loglog(np.arange(1, len(freq) + 1), np.maximum(freq, 1), color=PALETTE[0], linewidth=2)
    _style(ax, "Fig 1 - EC class size distribution (split{}, train)".format(s0),
           "EC label, ranked", "training sequences")
    ax.grid(True, which="both", **GRID)
    fig.tight_layout(); fig.savefig(fig_path("fig1_class_size_distribution.png")); FIGS.append("fig1")
    plt.close(fig)

# Figure 2 - the headline: level-4 micro-F1 against identity threshold.
d2 = METRICS[(METRICS.metric == "micro_f1") & (METRICS.level == 4) & (METRICS.test_variant == "official")]
if len(d2):
    fig, ax = plt.subplots(figsize=(5.6, 3.6))
    for i, (h, g) in enumerate(d2.groupby("head")):
        g = g.sort_values("split")
        ax.plot(g["split"], g["value"], marker="o", markersize=6, linewidth=2,
                color=PALETTE[i % len(PALETTE)], label=h)
        ax.annotate(h, (g["split"].iloc[-1], g["value"].iloc[-1]), textcoords="offset points",
                    xytext=(6, 0), fontsize=8, color=PALETTE[i % len(PALETTE)])
    _style(ax, "Fig 2 - level-4 micro-F1 vs sequence-identity threshold",
           "identity threshold of the split (%)", "micro-F1")
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout(); fig.savefig(fig_path("fig2_metric_vs_identity.png")); FIGS.append("fig2")
    plt.close(fig)

# Figure 3 - level-4 macro-F1 by training-frequency bin.
bin_names = [b[0] for b in BIN_EDGES]
d3 = METRICS[(METRICS.level == 4) & (METRICS.test_variant == "official")
             & (METRICS.metric.str.startswith("macro_f1["))]
if len(d3) and EMBED_SPLITS:
    s0 = max(EMBED_SPLITS)
    d3s = d3[d3.split == s0]
    heads = sorted(d3s["head"].unique())
    fig, ax = plt.subplots(figsize=(6.0, 3.6))
    w = 0.8 / max(len(heads), 1)
    for i, h in enumerate(heads):
        vals = []
        for b in bin_names:
            v = d3s[(d3s["head"] == h) & (d3s.metric == "macro_f1[{}]".format(b))]["value"]
            vals.append(float(v.iloc[0]) if len(v) else 0.0)
        ax.bar(np.arange(len(bin_names)) + i * w, vals, width=w * 0.92,
               color=PALETTE[i % len(PALETTE)], label=h)
    ax.set_xticks(np.arange(len(bin_names)) + 0.4 - w / 2); ax.set_xticklabels(bin_names)
    _style(ax, "Fig 3 - level-4 macro-F1 by training frequency (split{})".format(s0),
           "training sequences per EC", "macro-F1")
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout(); fig.savefig(fig_path("fig3_macro_f1_by_frequency_bin.png")); FIGS.append("fig3")
    plt.close(fig)

# Figure 4 - micro and macro F1 at each EC level.
d4 = METRICS[(METRICS.test_variant == "official") & (METRICS.metric.isin(["micro_f1", "macro_f1"]))]
if len(d4) and EMBED_SPLITS:
    s0 = max(EMBED_SPLITS)
    d4s = d4[d4.split == s0]
    heads = sorted(d4s["head"].unique())
    fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.4), sharey=True)
    for ax, met in zip(axes, ["micro_f1", "macro_f1"]):
        w = 0.8 / max(len(heads), 1)
        for i, h in enumerate(heads):
            g = d4s[(d4s["head"] == h) & (d4s.metric == met)].sort_values("level")
            ax.bar(g["level"].to_numpy() + i * w - 0.4, g["value"].to_numpy(), width=w * 0.92,
                   color=PALETTE[i % len(PALETTE)], label=h if met == "micro_f1" else None)
        _style(ax, "Fig 4 - {} by EC level (split{})".format(met.replace("_", "-"), s0),
               "EC level", met.replace("_", "-"))
        ax.set_xticks([1, 2, 3, 4])
    axes[0].legend(frameon=False, fontsize=8)
    fig.tight_layout(); fig.savefig(fig_path("fig4_metrics_by_level.png")); FIGS.append("fig4")
    plt.close(fig)

# Figure 5 - coverage against top-1 accuracy for the strongest head.
if EMBED_SPLITS and HEADS:
    s0 = max(EMBED_SPLITS); tag0 = MODEL_PASSES[0][1]
    key = (tag0, s0, "H3_mlp")
    if key in HEADS:
        space = get_label_space(s0)
        scorer = HEADS[key][0]
        feats = open_features(s0, "test", tag0)
        conf, correct = [], []
        for c in _iter_chunks(np.arange(space["n_test"]), EVAL_CHUNK):
            S = scorer.score_block(feats.rows(c))
            T = _dense_chunk(space["Yte"], c).astype(bool)
            a = S.argmax(axis=1)
            conf.append(S[np.arange(len(c)), a]); correct.append(T[np.arange(len(c)), a])
            del S, T
        conf = np.concatenate(conf); correct = np.concatenate(correct).astype(np.float64)
        o = np.argsort(-conf)
        cov = np.arange(1, len(o) + 1) / len(o)
        acc = np.cumsum(correct[o]) / np.arange(1, len(o) + 1)
        fig, ax = plt.subplots(figsize=(5.2, 3.4))
        ax.plot(cov, acc, color=PALETTE[0], linewidth=2)
        _style(ax, "Fig 5 - coverage vs top-1 accuracy (H3 MLP, split{})".format(s0),
               "fraction of test set predicted (coverage)", "top-1 accuracy")
        ax.set_ylim(0, 1)
        fig.tight_layout(); fig.savefig(fig_path("fig5_coverage_vs_accuracy.png")); FIGS.append("fig5")
        plt.close(fig)
        del feats; gc.collect()

print("figures written:", FIGS, "->", FIGURES_DIR)

In [ ]:
def _sha256(path, cap=400 * 1024 * 1024):
    h = hashlib.sha256()
    n = 0
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(1 << 20), b""):
            h.update(blk); n += len(blk)
            if n > cap:
                return h.hexdigest() + "-partial"
    return h.hexdigest()

MANIFEST = {
    "notebook": "EC_pipeline.ipynb",
    "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "smoke_test": SMOKE_TEST,
    "environment": ENV_REPORT,
    "seeds": {"python_numpy_torch": SEED},
    "flags": {"SPLITS_TO_RUN": list(SPLITS_TO_RUN), "FOLD": FOLD, "RUN_SPLIT100": RUN_SPLIT100,
              "ESM_MODEL_ID": ESM_MODEL_ID, "EMB_TAG": EMB_TAG,
              "SECOND_PASS_150M": SECOND_PASS_150M, "ESM_MODEL_ID_150M": ESM_MODEL_ID_150M,
              "MAX_LEN": MAX_LEN, "MAX_TOKENS_BATCH": MAX_TOKENS_BATCH, "SHARD_SIZE": SHARD_SIZE,
              "KEEP_SHARDS": KEEP_SHARDS, "RUN_LEGACY_FORENSIC": RUN_LEGACY_FORENSIC,
              "RUN_AAC_BASELINE": RUN_AAC_BASELINE, "RUN_DIAMOND": RUN_DIAMOND,
              "FORCE_RECOMPUTE": FORCE_RECOMPUTE, "HEAD_EPOCHS": HEAD_EPOCHS,
              "EFFECTIVE_EPOCHS": EFFECTIVE_EPOCHS, "HEAD_BATCH": HEAD_BATCH, "HEAD_LR": HEAD_LR,
              "HEAD_WEIGHT_DECAY": HEAD_WEIGHT_DECAY, "HEAD_HIDDEN": HEAD_HIDDEN,
              "HEAD_DROPOUT": HEAD_DROPOUT, "EARLY_STOP_PATIENCE": EARLY_STOP_PATIENCE,
              "VAL_FRACTION": VAL_FRACTION, "EVAL_CHUNK": EVAL_CHUNK},
    "stage_wall_clock_seconds": STAGE_TIMES,
    "input_files_sha256": {Path(p).name: _sha256(p) for p in sorted(INPUT_FILES)},
    "figures": FIGS,
}
mpath = res_path("run_manifest.json")
mpath.write_text(json.dumps(MANIFEST, indent=2))
print("manifest written ->", mpath)
print(json.dumps({"stage_wall_clock_seconds": STAGE_TIMES}, indent=2))

## Stage 14 — Limitations and supported claims

**What this notebook provides.** An integrity and exact-overlap audit, a forensic reconstruction of an earlier evaluation bug, and comparisons of composition features, frozen ESM-2 heads and a DIAMOND sequence-matching baseline. Saved main outputs cover fold 0 of the supplied split10/30/50/70 datasets. Results are reported by EC level and training-frequency group.

**Limits on interpretation.**

- No state-of-the-art or superiority claim over published CLEAN or other methods: no comparable local head-to-head is supplied.
- Frozen protein features with simple classifiers are established approaches. This implementation does not introduce a special class-balancing objective, sampler, or contrastive head.
- Test EC labels occur in the full supplied training label space, so these data do not establish unseen-EC prediction.
- Only one model fold is recorded. Training size and label space change with the supplied split settings, so the performance curve does not isolate sequence identity alone.
- Exact-duplicate checks do not replace an all-pairs similarity audit. No external or temporal evaluation is supplied for the main heads.
- The ESM-2 encoder is frozen. The embedding heads do not use structure or alignments; the separate DIAMOND baseline explicitly uses sequence alignments.
- Classifiers and centroids use the fit slice of training data, while DIAMOND searches the complete training reference. Independent per-level thresholds do not guarantee jointly nested binary predictions.

**Research artifacts and reproducibility.** Saved heads, label indices and thresholds are research outputs, not a deployed tool. Predictions for new raw sequences still require the encoder and preprocessing. The supplied source helper and CSV-reader repairs restore missing setup components and figure filters, but do not establish a fresh end-to-end GPU rerun or regenerate the archived results. Cached-stage timings are not fresh training or encoding benchmarks.


### Scientific references

These sources support the scientific background and method context. Published scores are not substituted for project results.

1. NC-IUBMB. [Enzyme Nomenclature](https://iubmb.qmul.ac.uk/enzyme/) and [How to Name and Classify Your Enzyme](https://iubmb.qmul.ac.uk/enzyme/advice.html).
2. Lin et al. (2023). Evolutionary-scale prediction of atomic-level protein structure with a language model. *Science*. ESM-2 representation context. [doi:10.1126/science.ade2574](https://doi.org/10.1126/science.ade2574).
3. Yu et al. (2023). Enzyme function prediction using contrastive learning. *Science*. CLEAN context and supplied benchmark provenance. [doi:10.1126/science.adf2465](https://doi.org/10.1126/science.adf2465).
4. Buchfink, Reuter and Drost (2021). Sensitive protein alignments at tree-of-life scale using DIAMOND. *Nature Methods*. [doi:10.1038/s41592-021-01101-x](https://doi.org/10.1038/s41592-021-01101-x).
5. Capela et al. (2025). Comparative Assessment of Protein Large Language Models for Enzyme Commission Number Prediction. *BMC Bioinformatics*. [doi:10.1186/s12859-025-06081-9](https://doi.org/10.1186/s12859-025-06081-9).
6. Ma et al. (2026). How not to be seen: predicting unseen enzyme functions using contrastive learning. *Bioinformatics*. EnzPlacer is context for a different unseen-label task, not a tested capability here. [doi:10.1093/bioinformatics/btag215](https://doi.org/10.1093/bioinformatics/btag215).


### Run complete

Generated artifacts are under the selected project's `outputs/` directory (or `outputs/smoke/` in smoke mode). Re-running reuses cached artifacts unless `FORCE_RECOMPUTE = True`.
